In [ ]:
!git clone https://github.com/MarioAlessandroNapoli/neuro-llm.git
%cd neuro-llm
!pip install -q -r requirements.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

In [ ]:
from huggingface_hub import snapshot_download, whoami

hf_user = whoami()["name"]
snapshot_download(f"{hf_user}/tinystories-tokenized", repo_type="dataset", local_dir="data")

In [ ]:
# Parametri espliciti per run: SEED e GROUP sono identita, non default.
# 5 seed baseline a 170M (D8): misurano sigma della val loss -> banda epsilon di D7.
# Config standard (bench 2026-08-18): DDP 2xT4, batch 16x2 = 32 globale, compile.
GROUP = 'grid-stage1'
for seed in [1, 2, 3, 4, 5]:
    !python -m src.train --arch transformer --tokens 170000000 --seed {seed} --lr 1e-3 \
        --devices 2 --batch-size 16 --compile --group {GROUP} \
        --hub-repo {hf_user}/neuro-llm-ckpt --max-time 00:11:00:00
